In [3]:
!pip install numba

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ------------------------------ --------- 2.1/2.7 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 11.4 MB/s  0:00:00
   ---------------------------------------- 0.0/38.1 MB ? eta -:--:--
   - -------------------------------------- 1.8/38.1 MB 12.6 MB/s eta 0:00:03
   ---- ----------------------------------- 4.5/38.1 MB 10.3 MB/s eta 0:00:04
   ------- -------------------------------- 6.8/38.1 MB 11.1 MB/s eta 0:00:03
   --------- ------------------------------ 9.4/38.1 MB 11.3 MB/s eta 0:00:03
   ------------ --------------------------- 11.8/38.1 MB 11.4 MB/s eta 0:00:03
   -------------- ------------------------- 13.9/38.1 MB 11.0 MB/s eta 0:00:03
   ----------------- ---------------------- 16.3/38.1 MB 11.3 MB/s eta 0:00:02
   ------------------- -------------------- 18.4/38.1 MB 11.0 MB/s eta 0:00:02
   --------------------- ------------------ 21.0/38.1 MB 11.1 MB/s eta 0:00:02
   

In [7]:
import os
from pathlib import Path
import numpy as np
import torch

from numba import cuda

In [5]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}") # type:ignore
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
CUDA version: 12.6
GPU: NVIDIA GeForce GTX 1650 Ti
Memory: 4.3 GB


## Using Numba Cuda

In [10]:
# Define the CUDA kernel using Python syntax
@cuda.jit
def vec_add_kernel(a, b, c):
    # Get the unique thread index in the 1D grid
    pos = cuda.grid(1)
    if pos < a.size:
        c[pos] = a[pos] + b[pos]

# Allocate data on host (CPU)
N = 100000
a = np.ones(N, dtype=np.float32)
b = np.ones(N, dtype=np.float32) * 2
c = np.zeros(N, dtype=np.float32)

# Copy data to device (GPU)
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.to_device(c)

# Launch the kernel: 256 threads per block, calculate blocks dynamically
threads_per_block = 256
blocks_per_grid = (N + (threads_per_block - 1)) // threads_per_block

vec_add_kernel[blocks_per_grid, threads_per_block](d_a, d_b, d_c)

# Copy results back to host
c = d_c.copy_to_host()
print("Success! First 5 elements:", c[:5])

Success! First 5 elements: [3. 3. 3. 3. 3.]


## Using Openai Triton

In [9]:
!pip install triton-windows

  Using cached triton_windows-3.7.0.post26-cp310-cp310-win_amd64.whl.metadata (1.8 kB)
Using cached triton_windows-3.7.0.post26-cp310-cp310-win_amd64.whl (49.6 MB)


In [11]:
import time
import os
from pathlib import Path

import torch

os.environ.setdefault("TRITON_CACHE_DIR", str(Path(".triton-cache").resolve()))

import triton
import triton.language as tl

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available on this machine/runtime.")

In [12]:
# CPU vs GPU benchmark using a Triton GPU kernel.
# This avoids the Windows cl.exe requirement from torch C++ extensions.

@triton.jit
def vector_add_kernel(a_ptr, b_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements

    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, a + b, mask=mask)

def triton_vector_add(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    out = torch.empty_like(a)
    n_elements = out.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    vector_add_kernel[grid](a, b, out, n_elements, BLOCK_SIZE=1024)
    return out

def time_cpu(fn, warmup=5, repeats=30):
    for _ in range(warmup):
        fn()

    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    end = time.perf_counter()
    return (end - start) * 1000 / repeats

def time_gpu(fn, warmup=10, repeats=100):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats

# Large enough to show GPU parallelism clearly.
n = 50_000_000

a_cpu = torch.rand(n, dtype=torch.float32)
b_cpu = torch.rand(n, dtype=torch.float32)

# Move data once before timing so the GPU timing measures computation, not PCIe transfer.
a_gpu = a_cpu.cuda()
b_gpu = b_cpu.cuda()

cpu_ms = time_cpu(lambda: a_cpu + b_cpu)
gpu_ms = time_gpu(lambda: triton_vector_add(a_gpu, b_gpu))

cpu_out = a_cpu + b_cpu
gpu_out = triton_vector_add(a_gpu, b_gpu)
max_diff = (cpu_out.cuda() - gpu_out).abs().max().item()

print(f"Elements: {n:,}")
print(f"CPU time: {cpu_ms:.3f} ms")
print(f"Triton GPU kernel time: {gpu_ms:.3f} ms")
print(f"Speedup: {cpu_ms / gpu_ms:.1f}x")
print(f"Max difference: {max_diff}")

Elements: 50,000,000
CPU time: 41.802 ms
Triton GPU kernel time: 3.375 ms
Speedup: 12.4x
Max difference: 0.0
